# Módulo 1 · Clase 11 — Programación Orientada a Objetos II

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Bloque | Qué aprenderás |
|---|--------|----------------|
| 1 | Herencia | Reutilizar y extender clases existentes |
| 2 | `super()` | Llamar al constructor y métodos del padre |
| 3 | Sobreescritura de métodos | Cambiar el comportamiento en la subclase |
| 4 | Herencia múltiple y MRO | Heredar de más de una clase |
| 5 | Polimorfismo | Misma interfaz, comportamientos distintos |
| 6 | Clases abstractas | Definir contratos con `ABC` |
| 7 | Encapsulación avanzada | `__slots__`, propiedades calculadas, patrones |
| 8 | Práctica | Sistema bancario |

> 💡 **Idea central:** la herencia nos permite construir sobre lo que ya existe sin reescribirlo. El polimorfismo nos permite tratar objetos distintos de forma uniforme. Juntos, hacen que el código sea extensible: añadir un nuevo tipo de cuenta bancaria no rompe nada de lo que ya funciona.

---
## 1. Herencia

La herencia permite que una clase (**subclase** o clase hija) herede atributos y métodos de otra clase (**superclase** o clase padre), añadiendo o modificando lo que necesite.

```
Clase padre (superclase)
     Animal
  nombre, energia
  comer(), dormir()
       │
       ├──────────────────┐
       ▼                  ▼
     Perro              Gato
  + raza             + indoor
  + ladrar()         + ronronear()
  (hereda comer,     (hereda comer,
   dormir)            dormir)
```

**Sintaxis:**
```python
class SubClase(SuperClase):
    ...
```

**Regla de oro:** usa herencia cuando la relación es «**es un**». Un `Perro` **es un** `Animal` ✅. Un `Perro` **tiene un** collar — eso es composición, no herencia.

In [ ]:
class Animal:
    def __init__(self, nombre, energia=100):
        self.nombre  = nombre
        self.energia = energia

    def comer(self, cantidad=20):
        self.energia += cantidad
        print(f"{self.nombre} come y recupera {cantidad} de energía (total: {self.energia})")

    def dormir(self, horas=8):
        recuperado    = horas * 5
        self.energia += recuperado
        print(f"{self.nombre} duerme {horas}h y recupera {recuperado} de energía (total: {self.energia})")

    def __str__(self):
        return f"{type(self).__name__}({self.nombre}, energía={self.energia})"


class Perro(Animal):          # Perro hereda de Animal
    def __init__(self, nombre, raza):
        super().__init__(nombre)  # llama al __init__ del padre
        self.raza = raza

    def ladrar(self):
        self.energia -= 5
        print(f"{self.nombre} ladra: ¡Guau! (energía: {self.energia})")


class Gato(Animal):
    def __init__(self, nombre, indoor=True):
        super().__init__(nombre)
        self.indoor = indoor

    def ronronear(self):
        print(f"{self.nombre} ronronea... 😸")


rex  = Perro("Rex",  "Pastor Alemán")
misi = Gato("Misi")

rex.comer()                    # heredado de Animal
rex.ladrar()                   # propio de Perro
misi.dormir(horas=12)          # heredado de Animal
misi.ronronear()               # propio de Gato

print()
print(rex)
print(misi)

# Comprobaciones de tipo
print(f"\nrex  es Animal: {isinstance(rex,  Animal)}")   # True
print(f"misi es Perro:  {isinstance(misi, Perro)}")    # False
print(f"Perro subclase de Animal: {issubclass(Perro, Animal)}")  # True

---
## 2. `super()` — Llamar al padre

`super()` devuelve un proxy al objeto padre. Lo usamos para:
1. Llamar al `__init__` del padre sin repetir su código.
2. Extender un método del padre en lugar de reemplazarlo.

```python
class Hijo(Padre):
    def __init__(self, a, b, c):
        super().__init__(a, b)   # inicializa lo del padre
        self.c = c               # añade lo propio

    def metodo(self):
        super().metodo()         # ejecuta el del padre primero
        # ... y luego añade comportamiento extra
```

In [ ]:
class Vehiculo:
    def __init__(self, marca, modelo, velocidad_max):
        self.marca         = marca
        self.modelo        = modelo
        self.velocidad_max = velocidad_max
        self.velocidad     = 0

    def acelerar(self, incremento):
        self.velocidad = min(self.velocidad + incremento, self.velocidad_max)
        print(f"{self.marca} {self.modelo}: {self.velocidad} km/h")

    def info(self):
        return f"{self.marca} {self.modelo} (max {self.velocidad_max} km/h)"


class Coche(Vehiculo):
    def __init__(self, marca, modelo, velocidad_max, puertas=4):
        super().__init__(marca, modelo, velocidad_max)  # delega al padre
        self.puertas = puertas

    def info(self):
        base = super().info()     # obtiene la info del padre
        return f"{base}, {self.puertas} puertas"  # y la extiende


class CocheElectrico(Coche):
    def __init__(self, marca, modelo, velocidad_max, bateria_kwh):
        super().__init__(marca, modelo, velocidad_max)  # llama a Coche.__init__
        self.bateria_kwh  = bateria_kwh
        self.carga        = 100   # porcentaje

    def acelerar(self, incremento):
        super().acelerar(incremento)   # comportamiento base
        consumo     = incremento * 0.1
        self.carga  = max(0, self.carga - consumo)
        print(f"  Batería: {self.carga:.1f}%")

    def info(self):
        base = super().info()
        return f"{base}, batería {self.bateria_kwh} kWh"


coche = Coche("Seat", "Ibiza", 180, puertas=5)
tesla = CocheElectrico("Tesla", "Model 3", 250, bateria_kwh=75)

print(coche.info())
coche.acelerar(100)

print()
print(tesla.info())
tesla.acelerar(80)
tesla.acelerar(120)

# La jerarquía de herencia
print(f"\nMRO de CocheElectrico:")
for cls in CocheElectrico.__mro__:
    print(f"  {cls.__name__}")

---
## 3. Sobreescritura de métodos

Una subclase puede **sobreescribir** cualquier método del padre para cambiar su comportamiento. Python siempre ejecuta el método más específico (el de la subclase).

```
objeto.metodo()
      │
      └─► Python busca en: SubClase → Padre → Abuelo → object
          Ejecuta el primero que encuentre
```

In [ ]:
class Figura:
    def area(self):
        raise NotImplementedError("Cada figura debe implementar area()")

    def descripcion(self):
        return f"{type(self).__name__} con área {self.area():.2f}"


class Circulo(Figura):
    def __init__(self, radio):
        self.radio = radio

    def area(self):            # sobreescribe el del padre
        import math
        return math.pi * self.radio ** 2


class Rectangulo(Figura):
    def __init__(self, ancho, alto):
        self.ancho = ancho
        self.alto  = alto

    def area(self):            # sobreescribe el del padre
        return self.ancho * self.alto


class Triangulo(Figura):
    def __init__(self, base, altura):
        self.base    = base
        self.altura  = altura

    def area(self):
        return (self.base * self.altura) / 2


figuras = [Circulo(5), Rectangulo(6, 4), Triangulo(3, 8)]

for fig in figuras:
    print(fig.descripcion())   # descripcion() está en Figura
                               # pero llama a area() de cada subclase

---
## 4. Herencia múltiple y MRO

Python permite heredar de más de una clase. El **MRO** (Method Resolution Order) define el orden en que Python busca métodos cuando hay herencia múltiple.

```python
class C(A, B):   # hereda de A y B
    pass
```

El MRO sigue el algoritmo **C3 linearization**: de izquierda a derecha, priorizando la subclase más específica. Puedes consultarlo con `Clase.__mro__` o `Clase.mro()`.

> En la práctica, la herencia múltiple se usa principalmente para **mixins** — clases pequeñas que añaden una funcionalidad concreta sin ser una jerarquía completa.

In [ ]:
# Patrón Mixin — clases que añaden funcionalidad sin ser jerarquías

class LogMixin:
    """Añade capacidad de logging a cualquier clase."""
    def log(self, mensaje):
        print(f"[LOG] {type(self).__name__}: {mensaje}")


class JsonMixin:
    """Añade serialización JSON a cualquier clase que tenga __dict__."""
    def to_json(self):
        import json
        datos = {k: v for k, v in self.__dict__.items() if not k.startswith("_")}
        return json.dumps(datos, ensure_ascii=False, indent=2)


class Producto:
    def __init__(self, nombre, precio):
        self.nombre = nombre
        self.precio = precio

    def aplicar_descuento(self, pct):
        self.precio = round(self.precio * (1 - pct / 100), 2)


class ProductoRico(LogMixin, JsonMixin, Producto):
    """Producto con logging y serialización JSON."""
    def __init__(self, nombre, precio, categoria):
        super().__init__(nombre, precio)
        self.categoria = categoria

    def aplicar_descuento(self, pct):
        self.log(f"Aplicando {pct}% de descuento sobre {self.precio} €")
        super().aplicar_descuento(pct)
        self.log(f"Precio final: {self.precio} €")


p = ProductoRico("Laptop", 1200, "Electrónica")
p.aplicar_descuento(15)
print()
print(p.to_json())
print()
print("MRO:", [c.__name__ for c in ProductoRico.__mro__])

---
## 5. Polimorfismo

El polimorfismo significa «muchas formas». En POO: **la misma llamada a un método produce resultados distintos según el tipo del objeto**.

No necesitas saber qué tipo concreto es un objeto para llamar a su método — basta con que todos compartan la misma interfaz.

```python
# Polimorfismo en acción
for figura in [Circulo(5), Rectangulo(4,6), Triangulo(3,8)]:
    print(figura.area())   # cada objeto responde a su manera
```

Python también tiene polimorfismo «duck typing»: si un objeto tiene el método que necesitas, da igual su tipo exacto (*«si camina como un pato y grazna como un pato, es un pato»*).

In [ ]:
# Polimorfismo via herencia
class Notificacion:
    def __init__(self, destinatario, mensaje):
        self.destinatario = destinatario
        self.mensaje      = mensaje

    def enviar(self):
        raise NotImplementedError


class Email(Notificacion):
    def enviar(self):
        print(f"📧 Email  → {self.destinatario}: {self.mensaje}")


class SMS(Notificacion):
    def enviar(self):
        print(f"📱 SMS    → {self.destinatario}: {self.mensaje[:160]}")


class PushNotification(Notificacion):
    def enviar(self):
        print(f"🔔 Push   → {self.destinatario}: {self.mensaje}")


def enviar_todas(notificaciones):
    """No sabe ni le importa el tipo concreto — solo llama a .enviar()."""
    for notif in notificaciones:
        notif.enviar()


cola = [
    Email("ana@mail.com",  "Tu pedido ha sido enviado"),
    SMS("+34 600 000 001",  "Código de verificación: 4821"),
    PushNotification("device_xyz", "Nueva oferta disponible"),
    Email("luis@mail.com", "Factura adjunta"),
]

enviar_todas(cola)

In [ ]:
# Duck typing — polimorfismo sin herencia
class ArchivoPDF:
    def leer(self):
        return "Contenido del PDF"

class ArchivoCSV:
    def leer(self):
        return "col1,col2\n1,2\n3,4"

class ArchivoJSON:
    def leer(self):
        return '{"clave": "valor"}'

# Esta función funciona con cualquier objeto que tenga .leer()
# No importa si heredan de una clase común o no
def procesar(archivo):
    contenido = archivo.leer()
    print(f"{type(archivo).__name__}: {len(contenido)} caracteres")

for archivo in [ArchivoPDF(), ArchivoCSV(), ArchivoJSON()]:
    procesar(archivo)

---
## 6. Clases abstractas con `ABC`

Una **clase abstracta** es una clase que no se puede instanciar directamente — solo sirve como plantilla para sus subclases. Define métodos que **deben** ser implementados por cualquier subclase.

```python
from abc import ABC, abstractmethod

class MiAbstracta(ABC):
    @abstractmethod
    def metodo_obligatorio(self):
        pass
```

Si una subclase no implementa todos los métodos abstractos, Python lanza `TypeError` al intentar instanciarla.

**¿Cuándo usar ABC vs `raise NotImplementedError`?**

| | `raise NotImplementedError` | `ABC + @abstractmethod` |
|-|----------------------------|--------------------------|
| El error aparece... | Al llamar al método | Al intentar instanciar |
| Puede instanciar la clase padre | Sí | No |
| Garantía de implementación | En runtime | En construcción |
| Cuándo usarlo | Scripts simples | Contratos firmes en librerías |

In [ ]:
from abc import ABC, abstractmethod

class CuentaBancaria(ABC):
    """Contrato base para todas las cuentas bancarias."""

    def __init__(self, titular, saldo_inicial=0):
        self.titular = titular
        self._saldo  = float(saldo_inicial)

    @property
    def saldo(self):
        return self._saldo

    @abstractmethod
    def depositar(self, cantidad):
        """Cada tipo de cuenta define sus propias reglas de depósito."""

    @abstractmethod
    def retirar(self, cantidad):
        """Cada tipo de cuenta define sus propias reglas de retirada."""

    @abstractmethod
    def tipo_cuenta(self):
        """Devuelve el nombre del tipo de cuenta."""

    # Método concreto disponible para todas las subclases
    def resumen(self):
        return f"[{self.tipo_cuenta()}] {self.titular} — Saldo: {self._saldo:,.2f} €"

    def __str__(self):
        return self.resumen()


# Intentar instanciar la clase abstracta directamente
try:
    cuenta = CuentaBancaria("Ana", 1000)
except TypeError as e:
    print(f"TypeError: {e}")

# Clase incompleta — no implementa todos los abstractos
class CuentaIncompleta(CuentaBancaria):
    def depositar(self, cantidad):
        self._saldo += cantidad
    # falta retirar() y tipo_cuenta()

try:
    c = CuentaIncompleta("Luis", 500)
except TypeError as e:
    print(f"TypeError (incompleta): {e}")

---
## 7. Encapsulación avanzada

Ampliamos los conceptos de encapsulación de la clase anterior con técnicas más precisas.

In [ ]:
# __slots__ — declara los atributos permitidos y ahorra memoria
class PuntoNormal:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    # Tiene __dict__ → se pueden añadir atributos nuevos

class PuntoSlots:
    __slots__ = ("x", "y")   # solo estos atributos están permitidos

    def __init__(self, x, y):
        self.x = x
        self.y = y

p_normal = PuntoNormal(1, 2)
p_slots  = PuntoSlots(3, 4)

p_normal.z = 99   # permitido
print(f"Normal: x={p_normal.x}, z={p_normal.z}, dict={p_normal.__dict__}")

try:
    p_slots.z = 99  # prohibido por __slots__
except AttributeError as e:
    print(f"Slots: {e}")

# Ventaja de __slots__: menos memoria (no hay __dict__)
import sys
print(f"\nTamaño PuntoNormal: {sys.getsizeof(p_normal.__dict__)} bytes (solo el dict)")
print(f"PuntoSlots no tiene __dict__: {'__dict__' not in dir(p_slots)}")

In [ ]:
# Propiedades calculadas y patron de validación encadenada
class Temperatura:
    def __init__(self, celsius):
        self.celsius = celsius   # usa el setter

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, valor):
        if valor < -273.15:
            raise ValueError(f"{valor}°C está por debajo del cero absoluto")
        self._celsius = float(valor)

    @property
    def fahrenheit(self):               # solo getter — calculado
        return self._celsius * 9/5 + 32

    @fahrenheit.setter
    def fahrenheit(self, valor):        # permite asignar en °F
        self.celsius = (valor - 32) * 5/9

    @property
    def kelvin(self):
        return self._celsius + 273.15

    def __str__(self):
        return f"{self._celsius:.1f}°C = {self.fahrenheit:.1f}°F = {self.kelvin:.2f}K"


t = Temperatura(100)
print(t)

t.fahrenheit = 32   # asignamos en Fahrenheit
print(t)            # se convierte automáticamente

try:
    t.celsius = -300
except ValueError as e:
    print(f"Error: {e}")

---
## 8. Práctica — Sistema bancario

Construimos un sistema bancario que utiliza todos los conceptos de POO II: herencia con `super()`, clases abstractas con `ABC`, polimorfismo, encapsulación y composición.

---

### Preparación — Arquitectura del sistema

```
CuentaBancaria (ABC)
  ├── depositar()  @abstractmethod
  ├── retirar()    @abstractmethod
  ├── tipo_cuenta() @abstractmethod
  └── resumen()    [concreto]
        │
        ├── CuentaCorriente
        │     sin límite de retirada, comisión por descubierto
        │
        ├── CuentaAhorro
        │     no permite descubierto, genera intereses
        │
        └── CuentaPlazoFijo
              bloqueada hasta fecha de vencimiento

Banco
  gestiona clientes y cuentas
  calcula estadísticas globales
```

In [ ]:
from abc import ABC, abstractmethod
from datetime import date, datetime

# ─── Excepciones del dominio bancario ────────────────────
class ErrorBancario(Exception):
    pass

class FondosInsuficientes(ErrorBancario):
    pass

class CuentaBloqueada(ErrorBancario):
    pass

class CantidadInvalida(ErrorBancario):
    pass

# ─── Clase base abstracta ─────────────────────────────────
class CuentaBancaria(ABC):
    _contador = 0

    def __init__(self, titular, saldo_inicial=0.0):
        CuentaBancaria._contador += 1
        self.numero   = f"ES{CuentaBancaria._contador:08d}"
        self.titular  = titular
        self._saldo   = 0.0
        self._historial = []
        if saldo_inicial > 0:
            self._registrar("Apertura", saldo_inicial)
            self._saldo = float(saldo_inicial)

    # ─── Propiedad saldo ─────────────────────────────────
    @property
    def saldo(self):
        return self._saldo

    # ─── Métodos abstractos ──────────────────────────────
    @abstractmethod
    def depositar(self, cantidad):
        pass

    @abstractmethod
    def retirar(self, cantidad):
        pass

    @abstractmethod
    def tipo_cuenta(self):
        pass

    # ─── Métodos concretos (compartidos por todas) ───────
    def _validar_cantidad(self, cantidad):
        if not isinstance(cantidad, (int, float)) or cantidad <= 0:
            raise CantidadInvalida(f"Cantidad inválida: {cantidad}. Debe ser un número positivo")

    def _registrar(self, concepto, importe, saldo_resultado=None):
        self._historial.append({
            "fecha":   datetime.now().strftime("%Y-%m-%d %H:%M"),
            "concepto": concepto,
            "importe":  importe,
            "saldo":    saldo_resultado if saldo_resultado is not None else self._saldo,
        })

    def historial(self, ultimas=None):
        movs = self._historial[-ultimas:] if ultimas else self._historial
        print(f"\n{'─'*60}")
        print(f"  Historial: {self.numero} — {self.titular}")
        print(f"{'─'*60}")
        print(f"  {'Fecha':<17} {'Concepto':<20} {'Importe':>10} {'Saldo':>10}")
        print(f"{'─'*60}")
        for m in movs:
            signo = "+" if m["importe"] >= 0 else ""
            print(f"  {m['fecha']:<17} {m['concepto']:<20} {signo}{m['importe']:>9.2f} {m['saldo']:>10.2f}")
        print(f"{'─'*60}")

    def resumen(self):
        return f"[{self.tipo_cuenta()}] {self.numero} — {self.titular} | {self._saldo:,.2f} €"

    def __str__(self):
        return self.resumen()

    def __repr__(self):
        return f"{type(self).__name__}(numero={self.numero!r}, titular={self.titular!r}, saldo={self._saldo})"

    def __eq__(self, otro):
        if not isinstance(otro, CuentaBancaria):
            return NotImplemented
        return self.numero == otro.numero

    def __lt__(self, otro):
        return self._saldo < otro._saldo


print("✅ Clase base CuentaBancaria definida")

---
### Ejercicio 1 — `CuentaCorriente` y `CuentaAhorro`

**Enunciado:** Implementa dos subclases de `CuentaBancaria`:

**`CuentaCorriente`:**
- Permite descubierto hasta un límite (`limite_descubierto`, por defecto 0).
- Si se va a descubierto, cobra una **comisión del 5%** sobre el importe retirado.
- `depositar` y `retirar` registran el movimiento en `_historial`.

**`CuentaAhorro`:**
- **No permite** saldo negativo — lanza `FondosInsuficientes`.
- Tiene un método `aplicar_interes(tasa_anual)` que abona los intereses y los registra.
- El depósito mínimo es 10 €.

**Ejemplo:**
```python
cc = CuentaCorriente("Ana García", saldo_inicial=500, limite_descubierto=200)
cc.retirar(600)    # descubierto: cobra comisión
ca = CuentaAhorro("Luis Pérez", saldo_inicial=1000)
ca.aplicar_interes(2.5)
```

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
class CuentaCorriente(CuentaBancaria):
    COMISION_DESCUBIERTO = 0.05   # 5%

    def __init__(self, titular, saldo_inicial=0.0, limite_descubierto=0.0):
        super().__init__(titular, saldo_inicial)
        self.limite_descubierto = float(limite_descubierto)

    def tipo_cuenta(self):
        return "Cuenta Corriente"

    def depositar(self, cantidad):
        self._validar_cantidad(cantidad)
        self._saldo += cantidad
        self._registrar("Ingreso", cantidad, self._saldo)
        return self._saldo

    def retirar(self, cantidad):
        self._validar_cantidad(cantidad)
        saldo_resultante = self._saldo - cantidad

        if saldo_resultante < -self.limite_descubierto:
            raise FondosInsuficientes(
                f"Fondos insuficientes. Saldo: {self._saldo:.2f} €, "
                f"límite de descubierto: {self.limite_descubierto:.2f} €"
            )

        self._saldo -= cantidad
        self._registrar("Retirada", -cantidad, self._saldo)

        # Cobrar comisión si entra en descubierto
        if self._saldo < 0:
            comision = round(abs(self._saldo) * self.COMISION_DESCUBIERTO, 2)
            self._saldo -= comision
            self._registrar("Comisión descubierto", -comision, self._saldo)
            print(f"⚠️  Descubierto. Comisión aplicada: {comision:.2f} €")

        return self._saldo


class CuentaAhorro(CuentaBancaria):
    DEPOSITO_MINIMO = 10.0

    def tipo_cuenta(self):
        return "Cuenta Ahorro"

    def depositar(self, cantidad):
        self._validar_cantidad(cantidad)
        if cantidad < self.DEPOSITO_MINIMO:
            raise CantidadInvalida(
                f"El depósito mínimo es {self.DEPOSITO_MINIMO:.2f} €"
            )
        self._saldo += cantidad
        self._registrar("Ingreso", cantidad, self._saldo)
        return self._saldo

    def retirar(self, cantidad):
        self._validar_cantidad(cantidad)
        if cantidad > self._saldo:
            raise FondosInsuficientes(
                f"Fondos insuficientes. Saldo: {self._saldo:.2f} €, "
                f"retirada solicitada: {cantidad:.2f} €"
            )
        self._saldo -= cantidad
        self._registrar("Retirada", -cantidad, self._saldo)
        return self._saldo

    def aplicar_interes(self, tasa_anual):
        if tasa_anual <= 0:
            raise CantidadInvalida("La tasa de interés debe ser positiva")
        interes = round(self._saldo * tasa_anual / 100, 2)
        self._saldo += interes
        self._registrar(f"Interés {tasa_anual}%", interes, self._saldo)
        print(f"💰 Interés aplicado: +{interes:.2f} € (tasa {tasa_anual}%)")        
        return interes


# ─── Pruebas ─────────────────────────────────────────────
CuentaBancaria._contador = 0

cc = CuentaCorriente("Ana García", saldo_inicial=500, limite_descubierto=200)
print(cc)

cc.depositar(300)
cc.retirar(400)
cc.retirar(500)   # entra en descubierto → comisión

try:
    cc.retirar(300)   # supera el límite
except FondosInsuficientes as e:
    print(f"❌ {e}")

cc.historial()

print()
ca = CuentaAhorro("Luis Pérez", saldo_inicial=1000)
print(ca)
ca.depositar(500)
ca.aplicar_interes(2.5)
ca.retirar(200)

try:
    ca.depositar(5)   # por debajo del mínimo
except CantidadInvalida as e:
    print(f"❌ {e}")

ca.historial()
```

**Puntos clave:**
- `super().__init__()` en ambas subclases delega la inicialización común (número, historial, validación del saldo inicial) al padre.
- `_validar_cantidad` está en el padre — las subclases la llaman sin duplicar la lógica.
- La comisión de descubierto se calcula **después** de actualizar `_saldo` — así el porcentaje se aplica sobre el importe descubierto real.
- `COMISION_DESCUBIERTO` y `DEPOSITO_MINIMO` son atributos de clase — fáciles de cambiar en un solo punto.
</details>

---
### Ejercicio 2 — `CuentaPlazoFijo`

**Enunciado:** Implementa `CuentaPlazoFijo(CuentaBancaria)`:

- Recibe `titular`, `saldo_inicial`, `tasa_anual` (ej: `3.5`) y `meses_plazo` (ej: `12`).
- Calcula y guarda `fecha_vencimiento` como `date`.
- **`retirar`** lanza `CuentaBloqueada` si no ha llegado la fecha de vencimiento.
- **`depositar`** lanza `CuentaBloqueada` siempre (no admite más depósitos tras la apertura).
- **`liquidar()`** — solo disponible tras el vencimiento — calcula el interés acumulado, lo suma al saldo y devuelve el total. Registra el movimiento.
- **`dias_restantes`** — propiedad que devuelve los días que quedan para el vencimiento (0 si ya venció).
- **`resumen`** — sobreescribe para incluir la fecha de vencimiento y la tasa.

**Ejemplo:**
```python
pf = CuentaPlazoFijo("Marta López", 10000, tasa_anual=3.5, meses_plazo=12)
print(pf.dias_restantes)   # ~365
pf.liquidar()              # CuentaBloqueada si no ha vencido
```

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
from datetime import date, timedelta


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
from datetime import date, timedelta

class CuentaPlazoFijo(CuentaBancaria):

    def __init__(self, titular, saldo_inicial, tasa_anual, meses_plazo):
        super().__init__(titular, saldo_inicial)
        self.tasa_anual        = float(tasa_anual)
        self.meses_plazo       = int(meses_plazo)
        self.fecha_apertura    = date.today()
        self.fecha_vencimiento = date.today() + timedelta(days=meses_plazo * 30)
        self._liquidado        = False

    def tipo_cuenta(self):
        return "Plazo Fijo"

    @property
    def dias_restantes(self):
        delta = (self.fecha_vencimiento - date.today()).days
        return max(0, delta)

    @property
    def ha_vencido(self):
        return date.today() >= self.fecha_vencimiento

    def depositar(self, cantidad):
        raise CuentaBloqueada(
            "Los depósitos adicionales no están permitidos en un plazo fijo"
        )

    def retirar(self, cantidad):
        if not self.ha_vencido:
            raise CuentaBloqueada(
                f"La cuenta vence el {self.fecha_vencimiento}. "
                f"Quedan {self.dias_restantes} días."
            )
        self._validar_cantidad(cantidad)
        if cantidad > self._saldo:
            raise FondosInsuficientes(f"Saldo disponible: {self._saldo:.2f} €")
        self._saldo -= cantidad
        self._registrar("Retirada", -cantidad, self._saldo)
        return self._saldo

    def liquidar(self):
        if not self.ha_vencido:
            raise CuentaBloqueada(
                f"No se puede liquidar hasta el {self.fecha_vencimiento} "
                f"({self.dias_restantes} días restantes)"
            )
        if self._liquidado:
            raise ErrorBancario("Esta cuenta ya fue liquidada")
        # Interés simple proporcional al plazo
        fraccion_año = self.meses_plazo / 12
        interes      = round(self._saldo * (self.tasa_anual / 100) * fraccion_año, 2)
        self._saldo += interes
        self._registrar(f"Liquidación {self.tasa_anual}% / {self.meses_plazo}m", interes, self._saldo)
        self._liquidado = True
        print(f"✅ Plazo fijo liquidado. Interés: +{interes:.2f} € → Saldo total: {self._saldo:,.2f} €")
        return self._saldo

    def resumen(self):
        estado = "VENCIDO" if self.ha_vencido else f"{self.dias_restantes}d restantes"
        return (
            f"[{self.tipo_cuenta()}] {self.numero} — {self.titular} | "
            f"{self._saldo:,.2f} € | {self.tasa_anual}% / {self.meses_plazo}m | {estado}"
        )


# ─── Pruebas ─────────────────────────────────────────────
pf = CuentaPlazoFijo("Marta López", 10_000, tasa_anual=3.5, meses_plazo=12)
print(pf)
print(f"Días restantes: {pf.dias_restantes}")

try:
    pf.depositar(500)
except CuentaBloqueada as e:
    print(f"❌ Depósito: {e}")

try:
    pf.retirar(1000)
except CuentaBloqueada as e:
    print(f"❌ Retirada: {e}")

try:
    pf.liquidar()
except CuentaBloqueada as e:
    print(f"❌ Liquidar: {e}")

# Simular que venció (forzar la fecha para la prueba)
pf.fecha_vencimiento = date.today() - timedelta(days=1)
print(f"\nSimulando vencimiento...")
print(f"¿Ha vencido? {pf.ha_vencido}")
pf.liquidar()
pf.historial()
```

**Puntos clave:**
- `ha_vencido` como `@property` encapsula la comparación de fechas — quien use la clase no necesita saber cómo se calcula.
- `_liquidado` evita que se aplique el interés dos veces si alguien llama a `liquidar()` en un bucle.
- `depositar` sobreescribe completamente el padre para **siempre** lanzar excepción — el plazo fijo no admite más depósitos.
- Forzar la fecha en las pruebas (`pf.fecha_vencimiento = ...`) es una técnica estándar para testear lógica temporal sin esperar días reales.
</details>

---
### Ejercicio 3 — Polimorfismo: operaciones sobre cualquier cuenta

**Enunciado:** Escribe funciones que trabajen de forma **polimórfica** sobre cualquier tipo de cuenta:

1. `calcular_patrimonio(cuentas)` → suma el saldo de todas las cuentas.
2. `cuenta_mas_rica(cuentas)` → devuelve la cuenta con mayor saldo.
3. `resumen_cartera(cuentas)` → imprime un informe agrupado por tipo de cuenta, con totales por tipo y total global.
4. `intentar_retirada(cuenta, cantidad)` → intenta retirar, captura las excepciones bancarias y devuelve `True/False`.

Estas funciones **no saben ni les importa** si la cuenta es corriente, ahorro o plazo fijo — solo llaman a la interfaz común.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
from collections import defaultdict

def calcular_patrimonio(cuentas):
    """Suma el saldo de todas las cuentas — polimorfismo puro."""
    return sum(c.saldo for c in cuentas)

def cuenta_mas_rica(cuentas):
    """Devuelve la cuenta con mayor saldo — usa __lt__ definido en CuentaBancaria."""
    if not cuentas:
        return None
    return max(cuentas)   # max() usa __lt__ → __gt__ internamente

def resumen_cartera(cuentas):
    """Informe agrupado por tipo — solo usa la interfaz de CuentaBancaria."""
    if not cuentas:
        print("La cartera está vacía.")
        return

    grupos = defaultdict(list)
    for c in cuentas:
        grupos[c.tipo_cuenta()].append(c)   # tipo_cuenta() es polimórfico

    total_global = calcular_patrimonio(cuentas)

    print(f"\n{'═'*62}")
    print(f"  RESUMEN DE CARTERA — {len(cuentas)} cuenta(s)")
    print(f"{'═'*62}")

    for tipo, grupo in sorted(grupos.items()):
        total_tipo = sum(c.saldo for c in grupo)
        print(f"\n  {tipo.upper()} ({len(grupo)}) — Total: {total_tipo:,.2f} €")
        print(f"  {'─'*58}")
        for c in sorted(grupo, reverse=True):   # ordenadas por saldo desc
            print(f"  {c.resumen()}")            # resumen() es polimórfico

    print(f"\n{'═'*62}")
    print(f"  PATRIMONIO TOTAL: {total_global:,.2f} €")
    print(f"{'═'*62}")

def intentar_retirada(cuenta, cantidad):
    """Intenta retirar. Devuelve True si tuvo éxito, False si no."""
    try:
        cuenta.retirar(cantidad)
        print(f"✅ Retirada de {cantidad:.2f} € de {cuenta.numero} — nuevo saldo: {cuenta.saldo:.2f} €")
        return True
    except CuentaBloqueada as e:
        print(f"🔒 Cuenta bloqueada ({cuenta.numero}): {e}")
        return False
    except FondosInsuficientes as e:
        print(f"💸 Fondos insuficientes ({cuenta.numero}): {e}")
        return False
    except CantidadInvalida as e:
        print(f"⚠️  Cantidad inválida: {e}")
        return False


# ─── Pruebas ─────────────────────────────────────────────
CuentaBancaria._contador = 0

cuentas = [
    CuentaCorriente("Ana García",  saldo_inicial=2500,  limite_descubierto=500),
    CuentaCorriente("Luis Pérez",  saldo_inicial=800),
    CuentaAhorro("Marta López",    saldo_inicial=5000),
    CuentaAhorro("Carlos Ruiz",    saldo_inicial=1200),
    CuentaPlazoFijo("Eva Martín",  10_000, tasa_anual=3.5, meses_plazo=12),
]

print(f"Patrimonio total: {calcular_patrimonio(cuentas):,.2f} €")
print(f"Cuenta más rica:  {cuenta_mas_rica(cuentas)}")

resumen_cartera(cuentas)

print("\n── Intentos de retirada ──")
intentar_retirada(cuentas[0], 1000)    # OK
intentar_retirada(cuentas[1], 5000)    # fondos insuficientes
intentar_retirada(cuentas[4], 1000)    # plazo fijo bloqueado
```

**Puntos clave:**
- `c.tipo_cuenta()`, `c.resumen()`, `c.retirar()` — cada llamada ejecuta el código de la subclase correcta. Las funciones no usan `isinstance()` ni `if type(c) == ...` — eso rompería el polimorfismo.
- `max(cuentas)` funciona porque `CuentaBancaria` define `__lt__` — Python puede comparar cualquier par de cuentas sin saber su tipo.
- `intentar_retirada` captura `ErrorBancario` (base) en subclases específicas — el orden de los `except` importa: el más específico primero.
- `defaultdict(list)` agrupa las cuentas por tipo sin necesidad de comprobar si la clave existe.
</details>

---
### Ejercicio 4 — La clase `Banco`

**Enunciado:** Crea la clase `Banco` que gestiona clientes y cuentas:

- **`abrir_cuenta(tipo, titular, **kwargs)`** — crea una cuenta del tipo indicado (`"corriente"`, `"ahorro"`, `"plazo_fijo"`), la guarda y la devuelve.
- **`buscar_cuenta(numero)`** — devuelve la cuenta o lanza `KeyError`.
- **`transferir(numero_origen, numero_destino, cantidad)`** — mueve dinero entre cuentas. Usa `retirar` y `depositar` — el polimorfismo se encarga del resto.
- **`total_depositos()`** — suma de todos los saldos positivos.
- **`informe()`** — llama a `resumen_cartera` del ejercicio anterior.
- **`__len__`** — número de cuentas.
- **`__contains__`** — permite `numero in banco`.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
class Banco:
    TIPOS = {
        "corriente":  CuentaCorriente,
        "ahorro":     CuentaAhorro,
        "plazo_fijo": CuentaPlazoFijo,
    }

    def __init__(self, nombre):
        self.nombre   = nombre
        self._cuentas = {}   # {numero: CuentaBancaria}

    def abrir_cuenta(self, tipo, titular, **kwargs):
        if tipo not in self.TIPOS:
            raise ValueError(f"Tipo de cuenta desconocido: '{tipo}'. Opciones: {list(self.TIPOS)}")
        ClaseCuenta = self.TIPOS[tipo]
        cuenta = ClaseCuenta(titular, **kwargs)
        self._cuentas[cuenta.numero] = cuenta
        print(f"✅ Cuenta abierta: {cuenta}")
        return cuenta

    def buscar_cuenta(self, numero):
        cuenta = self._cuentas.get(numero)
        if cuenta is None:
            raise KeyError(f"No existe ninguna cuenta con número '{numero}'")
        return cuenta

    def transferir(self, numero_origen, numero_destino, cantidad):
        origen  = self.buscar_cuenta(numero_origen)
        destino = self.buscar_cuenta(numero_destino)
        # El polimorfismo maneja las reglas de cada tipo de cuenta
        origen.retirar(cantidad)
        destino.depositar(cantidad)
        print(
            f"🔄 Transferencia: {cantidad:.2f} € "
            f"de {origen.numero} ({origen.titular}) "
            f"a {destino.numero} ({destino.titular})"
        )

    def total_depositos(self):
        return sum(c.saldo for c in self._cuentas.values())

    def informe(self):
        print(f"\n  Banco: {self.nombre}")
        resumen_cartera(list(self._cuentas.values()))

    def __len__(self):
        return len(self._cuentas)

    def __contains__(self, numero):
        return numero in self._cuentas

    def __str__(self):
        return f"{self.nombre} — {len(self)} cuenta(s), {self.total_depositos():,.2f} € en depósitos"


# ─── Pruebas ─────────────────────────────────────────────
CuentaBancaria._contador = 0

banco = Banco("Banco Digital Galicia")

c1 = banco.abrir_cuenta("corriente",  "Ana García",  saldo_inicial=3000, limite_descubierto=500)
c2 = banco.abrir_cuenta("ahorro",     "Luis Pérez",  saldo_inicial=8000)
c3 = banco.abrir_cuenta("corriente",  "Marta López", saldo_inicial=1500)
c4 = banco.abrir_cuenta("plazo_fijo", "Carlos Ruiz", saldo_inicial=20_000, tasa_anual=4.0, meses_plazo=24)

print(f"\nCuentas en el banco: {len(banco)}")
print(f"{c1.numero} en banco: {c1.numero in banco}")

print("\n── Transferencia ──")
try:
    banco.transferir(c1.numero, c2.numero, 1000)
except ErrorBancario as e:
    print(f"❌ {e}")

# Transferir a plazo fijo → CuentaBloqueada
print()
try:
    banco.transferir(c1.numero, c4.numero, 500)
except CuentaBloqueada as e:
    print(f"❌ No se puede transferir a plazo fijo: {e}")

banco.informe()
```

**Puntos clave:**
- `TIPOS = {"corriente": CuentaCorriente, ...}` es el patrón **factory** — el diccionario mapea un string a una clase. `ClaseCuenta(titular, **kwargs)` crea la instancia sin un `if/elif` largo.
- `transferir` no hace ningún `isinstance` ni `if tipo == ...` — llama a `retirar` y `depositar`, y el polimorfismo se encarga. Si el plazo fijo rechaza el depósito, la excepción se propaga sola.
- `buscar_cuenta` usa `.get()` y lanza `KeyError` — mismo comportamiento que un diccionario nativo, consistente con las expectativas de quien usa la clase.
</details>

---
### Ejercicio 5 (reto) — `CuentaJoven` y herencia en cadena

**Enunciado:** Crea `CuentaJoven(CuentaAhorro)` que hereda de `CuentaAhorro` y añade:

- Solo se puede abrir para titulares con edad entre 14 y 25 años (recibe `edad` en el constructor).
- El depósito mínimo baja a 5 € (sobreescribe la constante de la clase padre).
- `retirar` aplica un **límite mensual de 500 €** — acumula lo retirado en el mes actual y lanza `ErrorBancario` si se supera.
- `aplicar_interes` aplica una tasa un **50% mayor** que la de ahorro normal (bonus joven).
- `cumplir_anios()` — si el titular cumple 26, convierte la cuenta a `CuentaAhorro` normal (devuelve una nueva instancia con el mismo saldo e historial).

Integra `CuentaJoven` en el sistema del `Banco` añadiendo `"joven"` al diccionario `TIPOS`.

Construye un escenario completo y demuestra la cadena de herencia:
```
CuentaBancaria (ABC)
    └── CuentaAhorro
            └── CuentaJoven
```

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
from datetime import date

class CuentaJoven(CuentaAhorro):
    DEPOSITO_MINIMO  = 5.0    # sobreescribe el de CuentaAhorro (10.0)
    LIMITE_MENSUAL   = 500.0
    EDAD_MINIMA      = 14
    EDAD_MAXIMA      = 25
    BONUS_INTERES    = 1.5    # 50% más que la tasa base

    def __init__(self, titular, edad, saldo_inicial=0.0):
        if not (self.EDAD_MINIMA <= edad <= self.EDAD_MAXIMA):
            raise ValueError(
                f"La cuenta joven es para personas de {self.EDAD_MINIMA} "
                f"a {self.EDAD_MAXIMA} años. Edad recibida: {edad}"
            )
        super().__init__(titular, saldo_inicial)
        self.edad          = edad
        self._retirado_mes = 0.0
        self._mes_actual   = date.today().month

    def tipo_cuenta(self):
        return "Cuenta Joven"

    def _resetear_limite_si_nuevo_mes(self):
        mes_hoy = date.today().month
        if mes_hoy != self._mes_actual:
            self._retirado_mes = 0.0
            self._mes_actual   = mes_hoy

    def retirar(self, cantidad):
        self._resetear_limite_si_nuevo_mes()
        if self._retirado_mes + cantidad > self.LIMITE_MENSUAL:
            disponible = self.LIMITE_MENSUAL - self._retirado_mes
            raise ErrorBancario(
                f"Límite mensual de {self.LIMITE_MENSUAL:.2f} € alcanzado. "
                f"Ya has retirado {self._retirado_mes:.2f} €, "
                f"te quedan {disponible:.2f} € disponibles este mes."
            )
        super().retirar(cantidad)   # delega la lógica de fondos al padre
        self._retirado_mes += cantidad

    def aplicar_interes(self, tasa_anual):
        tasa_con_bonus = round(tasa_anual * self.BONUS_INTERES, 4)
        print(f"🎁 Bonus joven: tasa {tasa_anual}% → {tasa_con_bonus}%")
        return super().aplicar_interes(tasa_con_bonus)

    def cumplir_anios(self):
        self.edad += 1
        if self.edad > self.EDAD_MAXIMA:
            print(f"🎂 {self.titular} ha cumplido {self.edad} años. "
                  f"La cuenta joven se convierte a cuenta ahorro estándar.")
            nueva = CuentaAhorro(self.titular, saldo_inicial=0.0)
            nueva._saldo     = self._saldo        # transferir saldo
            nueva._historial = self._historial[:]  # copiar historial
            nueva._registrar("Conversión desde Cuenta Joven", 0, nueva._saldo)
            return nueva
        print(f"🎂 {self.titular} cumple {self.edad} años. Cuenta joven activa.")
        return self

    def resumen(self):
        retirado = self._retirado_mes
        base     = super().resumen()
        return f"{base} | Edad: {self.edad} | Retirado este mes: {retirado:.2f}/{self.LIMITE_MENSUAL:.0f} €"


# Integrar en Banco
Banco.TIPOS["joven"] = CuentaJoven


# ─── Escenario completo ───────────────────────────────────
CuentaBancaria._contador = 0
banco = Banco("Banco Digital Galicia")

cj = banco.abrir_cuenta("joven", "Sofía Ramos", edad=20, saldo_inicial=800)

print("\n── Operaciones cuenta joven ──")
cj.depositar(50)          # mínimo 5 € (no 10 como en CuentaAhorro)
cj.retirar(200)
cj.retirar(250)

try:
    cj.retirar(100)       # supera el límite mensual de 500 €
except ErrorBancario as e:
    print(f"❌ {e}")

cj.aplicar_interes(2.0)   # recibirá 3.0% gracias al bonus
print(cj)

# Verificar la cadena de herencia
print("\n── Cadena de herencia ──")
print(f"MRO: {[c.__name__ for c in CuentaJoven.__mro__]}")
print(f"isinstance(cj, CuentaAhorro):   {isinstance(cj, CuentaAhorro)}")
print(f"isinstance(cj, CuentaBancaria): {isinstance(cj, CuentaBancaria)}")

# Cumplir años y convertir
print("\n── Cumplir años ──")
for _ in range(6):                  # de 20 a 26
    cj = cj.cumplir_anios()

print(f"Tipo tras conversión: {type(cj).__name__}")
cj.historial(ultimas=5)
```

**Puntos clave:**
- `DEPOSITO_MINIMO = 5.0` en `CuentaJoven` sobreescribe el atributo de clase del padre sin tocar el padre — la herencia de atributos de clase funciona igual que la de métodos.
- `super().retirar(cantidad)` en `CuentaJoven.retirar` delega la lógica de saldo negativo a `CuentaAhorro` — no duplicamos esa validación.
- `cumplir_anios` devuelve `self` o una nueva instancia de `CuentaAhorro` — el código que llama solo guarda el retorno (`cj = cj.cumplir_anios()`) y no le importa cuál de los dos recibe.
- `Banco.TIPOS["joven"] = CuentaJoven` extiende el sistema sin modificar la clase `Banco` — abierto a extensión, cerrado a modificación (principio Open/Closed).
</details>

---
## Resumen de la clase

### Herencia

```python
class Hijo(Padre):                    # hereda todo del padre
    def __init__(self, a, b, c):
        super().__init__(a, b)        # inicializa lo del padre
        self.c = c                    # añade lo propio

    def metodo(self):                 # sobreescribe el del padre
        resultado = super().metodo()  # puede llamar al padre primero
        return resultado + " extra"
```

### Clases abstractas

```python
from abc import ABC, abstractmethod

class Base(ABC):
    @abstractmethod
    def metodo_obligatorio(self):   # las subclases DEBEN implementarlo
        pass

    def metodo_concreto(self):      # disponible para todas las subclases
        return self.metodo_obligatorio()  # puede llamar a los abstractos
```

### Tabla de referencia: relaciones entre clases

| Relación | Cuándo | Ejemplo |
|----------|--------|---------|
| **Herencia** (`es un`) | La subclase es una versión especializada | `CuentaJoven` es una `CuentaAhorro` |
| **Composición** (`tiene un`) | Una clase usa objetos de otra | `Banco` tiene `CuentaBancaria`s |
| **Mixin** | Añadir funcionalidad transversal | `LogMixin`, `JsonMixin` |

### Polimorfismo — señales de buen diseño

✅ El código que llama **no usa `isinstance()`** para decidir qué hacer  
✅ Añadir un nuevo tipo **no requiere modificar** el código existente  
✅ La interfaz común (métodos abstractos) **garantiza el contrato**

---

## ¿Qué viene en la próxima clase?

En la **Clase 12** cerramos el Módulo 1 con un **proyecto integrador** que usará todo lo aprendido: estructuras de datos, funciones, archivos, excepciones y POO. Construiremos un sistema completo de principio a fin.